In [1]:
import re
import pandas as pd

# 1. Skill dictionary: canonical name -> regex pattern covering variants

SKILL_PATTERNS = {
    "Python":            r"python\s*3?\.?\d*",                  # Python, Python 3, Python3, Python 3.9
    "SQL":               r"(my\s?sql|postgre\s?sql|sql\s*server|sql)",  # sql, mysql, postgresql, sql server
    "Java":              r"java(?!\s*script)",                  # java, but not javascript
    "JavaScript":        r"java\s*script|(?<!\.)\bjs\b",         # javascript, java script, standalone JS (not the .js in React.js/Node.js)
    "Power BI":          r"power\s*bi",                          # power bi, powerbi
    "Tableau":           r"tableau",
    "Excel":             r"(ms\s*excel|microsoft\s*excel|excel)",
    "AWS":               r"(aws|amazon\s+web\s+services)",
    "Azure":             r"(microsoft\s+)?azure",
    "GCP":               r"(gcp|google\s+cloud(\s+platform)?)",
    "Kubernetes":        r"kubernetes|k8s",
    "Docker":            r"docker",
    "Git":               r"\bgit\b(?!hub)",                      # git, but not github
    "CI/CD":             r"ci\s*/\s*cd|continuous\s+integration",
    "Spring Boot":       r"spring\s*boot",
    "React":             r"react(\.?js)?",                       # react, reactjs, react.js
    "Node.js":           r"node(\.?js)?",                        # node, nodejs, node.js
    "TensorFlow":        r"tensor\s*flow",
    "PyTorch":           r"py\s*torch",
    "Pandas":            r"\bpandas\b",
    "NumPy":             r"num\s*py",
    "Scikit-learn":      r"scikit[\s-]?learn|sklearn",
    "Figma":             r"figma",
    "Machine Learning":  r"machine\s+learning|\bml\b",
    "Deep Learning":     r"deep\s+learning|\bdl\b",
    "NLP":               r"\bnlp\b|natural\s+language\s+processing",
    "SAP":               r"\bsap\b",
    "Jira":              r"jira",
    "Agile":             r"agile",
    "Scrum":             r"scrum",
    "R":                 r"(?<![a-zA-Z])r(?![a-zA-Z])(?=\s*(language|programming|studio)?\b)",
}

# Pre-compile once, with word boundaries wrapped around the whole pattern
# so e.g. "python" doesn't match inside "pythonic-sounding-word".
COMPILED_PATTERNS = {
    skill: re.compile(rf"(?<![\w-])(?:{pattern})(?![\w-])", re.IGNORECASE)
    for skill, pattern in SKILL_PATTERNS.items()
}


def extract_skills(text: str) -> list[str]:
    """Search text for every skill pattern; return canonical skill names
    (deduplicated) regardless of which surface form matched."""
    if not isinstance(text, str) or not text:
        return []
    found = []
    for skill, pattern in COMPILED_PATTERNS.items():
        if pattern.search(text):
            found.append(skill)
    return found


# 2. Self-test: confirm variant forms normalize correctly

def _run_self_tests():
    cases = [
        ("Experience with Python required.", "Python"),
        ("Must know Python 3 well.", "Python"),
        ("Comfortable coding in Python3.", "Python"),
        ("Strong Python programming skills.", "Python"),
        ("Familiar with React.js and Node.js.", "React"),
        ("Knows plain JS well.", "JavaScript"),
        ("Experience with JavaScript.", "JavaScript"),
        ("Uses PowerBI for dashboards.", "Power BI"),
        ("Skilled in MySQL and PostgreSQL.", "SQL"),
        ("Knows JavaScript and TypeScript.", "JavaScript"),
    ]
    print("=== Self-test: variant normalization ===")
    for text, expected_skill in cases:
        found = extract_skills(text)
        status = "PASS" if expected_skill in found else "FAIL"
        print(f"{status}: {text!r} -> {found}")
    print()


# 3. Run on the real dataset

def main():
    _run_self_tests()

    df = pd.read_csv("C:/Users/HP/Desktop/Internship Final Submission/NLP/cleaned_job_dataset after EDA.csv")
    df["extracted_skills"] = df["job_description"].apply(extract_skills)
    df["skill_count"] = df["extracted_skills"].apply(len)

    df.to_csv("job_dataset_with_skills_v2.csv", index=False)

    print(f"Processed {len(df)} job descriptions.")
    print("Saved -> job_dataset_with_skills_v2.csv")
    print()
    print("Sample output:")
    print(df[["job_id", "job_title", "extracted_skills"]].head(10).to_string(index=False))

    print()
    print("Top 15 most in-demand skills (normalized):")
    all_skills = [s for skills in df["extracted_skills"] for s in skills]
    print(pd.Series(all_skills).value_counts().head(15).to_string())


if __name__ == "__main__":
    main()

=== Self-test: variant normalization ===
PASS: 'Experience with Python required.' -> ['Python']
PASS: 'Must know Python 3 well.' -> ['Python']
PASS: 'Comfortable coding in Python3.' -> ['Python']
PASS: 'Strong Python programming skills.' -> ['Python']
PASS: 'Familiar with React.js and Node.js.' -> ['React', 'Node.js']
PASS: 'Knows plain JS well.' -> ['JavaScript']
PASS: 'Experience with JavaScript.' -> ['JavaScript']
PASS: 'Uses PowerBI for dashboards.' -> ['Power BI']
PASS: 'Skilled in MySQL and PostgreSQL.' -> ['SQL']
PASS: 'Knows JavaScript and TypeScript.' -> ['JavaScript']

Processed 1000 job descriptions.
Saved -> job_dataset_with_skills_v2.csv

Sample output:
 job_id                        job_title                                           extracted_skills
 100001         Supply Chain Coordinator                                                         []
 100002        Backend Software Engineer           [Java, AWS, Kubernetes, Git, CI/CD, Spring Boot]
 100003                  